# Week 8.5 — frozen sample-efficiency confirmation

This compact notebook reads the locked, machine-readable outputs. It does not rerun or tune acquisition methods.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'week_08_5' else Path.cwd()
OUT = ROOT / 'outputs' / 'week8_5_frozen_confirmation'
protocol = json.loads((OUT/'preregistered_protocol.json').read_text())
protocol['protocol_id']

'week8_5_frozen_confirmation_protocol/v1.0.0'

## 1. What was frozen?

The population, manual `has_keyhole` label, four inputs (`P`, `VX`, `LS`, `ST`), grouped splits, three acquisition arms, integer AULC orientation, adaptive horizon rule, and claim thresholds were locked before confirmation.

In [2]:
splits = pd.read_csv(OUT/'split_manifest.csv')
initial = pd.read_csv(OUT/'initial_design_manifest.csv')
print(splits.groupby('role').size())
initial.groupby('run_id').agg(rows=('query_order','count'), keyholes=('revealed_has_keyhole','sum')).head()

role
training_pool     32400
untouched_test     8100
dtype: int64


,rows,keyholes
run_id,,
w85__r01_f01,16,3
w85__r01_f02,16,6
w85__r01_f03,16,6
w85__r01_f04,16,6
w85__r01_f05,16,3


## 2. How to read the primary result

The primary contrast is margin AULC minus the mean of 30 matched Random continuations inside each fold. Inference resamples repeat blocks, keeps five folds together, and resamples Random continuations within folds.

In [3]:
ledger = pd.read_csv(OUT/'claim_decisions.csv')
ledger[['claim','decision','point_estimate','one_sided_95pct_lower_bound','horizon']]

,claim,decision,point_estimate,one_sided_95pct_lower_bound,horizon
0,primary_performance,PASS,0.037300,0.031270,160
1,query_saving,QUALIFY,20.099000,13.909950,160
2,multiplier,QUALIFY,1.514436,1.329210,160
3,repulsion_h_0.15,QUALIFY,0.001829,-0.000625,160
4,overall_primary_confirmation,NOT_CONFIRMED,NaN,NaN,160


## 3. Censoring and interpretation

A missing persistent crossing is reported as right-censored at `H+`. Restricted-horizon estimands use `H` only for the declared burden calculation; simple finite crossing summaries never impute an unobserved query count. Results concern this saved-simulation population and are neither causal claims nor physical-boundary certainty.

In [4]:
cross = pd.read_csv(OUT/'query_crossings.csv')
cross.query('target == 0.8').groupby('arm').agg(event_rate=('event_observed','mean'), finite_median=('crossing_budget','median'))

,event_rate,finite_median
arm,,
binary_margin,0.910,22.0
binary_random,0.831,24.0
binary_uncertainty_repulsion,0.910,22.0
